# Activation Steering — 파트 2 (v4, **모델-적응형**): 큰 모델 재현

같은 대조쌍(`pairs_final.jsonl`)으로 **더 큰 모델**에서 추출 → 4a → 4b(자기보고 2채점+접지) 재현.
모델·레이어·양자화를 **환경변수 하나로** 바꿉니다 (코드는 모델-적응형).

**권장(우선순위 1+2 정합)**: `gemma-2-9b-it` (SAE 스텝2가 Gemma Scope로 그대로 됨).
**GPU**: 9B bf16 ≈ 18GB → Colab **Pro(L4/A100)**. 무료 T4(16GB)면 `STEER_4BIT=1`.


## 1. GPU + 의존성 + HF 로그인


In [ ]:
!nvidia-smi -L

In [ ]:
!pip install -q "transformers>=4.42" accelerate huggingface_hub openai python-dotenv bitsandbytes
import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # gemma-2-9b-it 라이선스도 수락 필요

## 2. 번들 업로드 + 모델 설정
`colab_bundle_4b.zip` 업로드 후, 아래에서 모델/레이어 지정.


In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()   # colab_bundle_4b.zip
name = next(iter(up))
os.makedirs('/content/asteer', exist_ok=True)
with zipfile.ZipFile(name) as z: z.extractall('/content/asteer')
%cd /content/asteer

### 모델 설정 (여기만 바꾸면 됨)


In [ ]:
import os
os.environ['STEER_MODEL'] = 'google/gemma-2-9b-it'   # 9B. 2B로 되돌리려면 google/gemma-2-2b-it
os.environ['STEER_LAYER'] = '20'                      # Gemma Scope 9B SAE 층. (2B면 이 줄 삭제)
# os.environ['STEER_4BIT'] = '1'                      # 무료 T4(16GB)면 주석 해제(4-bit)
print('MODEL =', os.environ['STEER_MODEL'], '| LAYER =', os.environ.get('STEER_LAYER'),
      '| 4BIT =', os.environ.get('STEER_4BIT','off'))

## 3. 추출 → 4. 벡터 구성 + 4a 진단
환경변수(STEER_MODEL/LAYER/4BIT)를 그대로 상속. 추출은 모델 차원 자동 감지.


In [ ]:
!python steering/extract_activations.py

In [ ]:
!python steering/build_vectors.py
!python steering/diagnostics.py --layers ${STEER_LAYER:-12}

## 5. 4b 스티어링 평가 (자기보고 2채점 + 접지)


In [ ]:
!python steering/steer_eval.py --smoke

In [ ]:
!python steering/steer_eval.py

## 6. 결과 보기 + 다운로드


In [ ]:
import json, glob
L = os.environ.get('STEER_LAYER','12')
r = json.load(open('artifacts/vectors/steer_eval.json'))
print('모델:', os.environ['STEER_MODEL'], '| layer', r['layer'], '| R', round(r['R'],1))
for s in r['sweep']:
    e = ' '.join(f"{k}={s[k]}" for k in s if k[:2] in ('E_','N_'))
    print(f"  c={s['c']:+.2f} | {e} | bproj={s['behavior_proj']:+.2f} rep={s['repetition']:.2f}")
print('\n요약:', json.dumps(r['summary'], ensure_ascii=False, indent=2))
for f in glob.glob('artifacts/vectors/*.json'): files.download(f)

---
**판정**: 큰 모델에서 요약 `self_report` 의 외향 dose-corr·ΔE 가 커지면 → 2B 용량 문제였던 것.
여전히 평평(ΔE≈0)하면 → 모델 크기 무관한 (b)/(c) → 다음은 SAE(스텝2) + A/B/C/D 축 비교(스텝3).

`steer_eval.json`(+layer meta) 공유 바람.
